# Transfer Learning v2 – Tối ưu cho Vietnam Data

### 🔧 Changelog v3.1 (Low-Data Regime & Anti-Overfit):
1. ✅ **Khôi phục Nguyên trạng Normalizer**: Đưa lại `transformation='softplus'` để bảo toàn sự khớp nối cấu trúc Transfer Learning với pretrained weights gốc.
2. ✅ **Tối đa hóa Lượng mẫu học (Data-mining)**: Hạ `min_encoder_length` xuống `12` để chắt chiu thêm từng cửa sổ trượt (windows) thời gian từ tập dữ liệu siêu nhỏ (72 tháng).
3. ✅ **Tăng cường Phòng ngự Overfitting**: Đẩy `dropout` lên `0.25` và `weight_decay` lên `1e-2`. Xóa bỏ Stage 4 dư thừa tránh để mô hình học vẹt tập huấn luyện.
4. ✅ **Phục hồi Chức năng Vẽ biểu đồ**: Bổ sung lại các Cell vẽ biểu đồ trực quan như bạn yêu cầu.

*(Hãy "Run All" notebook này để bắt đầu)*

In [27]:
# ── Cell 1: Import ────────────────────────────────────────────────────────────
import os, json, shutil, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import r2_score

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

pl.seed_everything(42, workers=True)
torch.set_float32_matmul_precision('high')

config_path = Path('checkpoint/model_config_v2.json')
if config_path.exists():
    with open(config_path, 'r') as f:
        model_cfg = json.load(f)
    LOG_TRANSFORM = model_cfg.get('LOG_TRANSFORM', True)
    print(f'✅ Config loaded – LOG_TRANSFORM={LOG_TRANSFORM}')
else:
    LOG_TRANSFORM = True
    print('⚠️ model_config_v2.json not found, LOG_TRANSFORM=True')

CHECKPOINT_PATH = 'checkpoint/tft_v2_best.ckpt'
print(f'Pretrained: {CHECKPOINT_PATH} | Exists: {os.path.exists(CHECKPOINT_PATH)}')

Seed set to 42


✅ Config loaded – LOG_TRANSFORM=True
Pretrained: checkpoint/tft_v2_best.ckpt | Exists: True


In [28]:
# ── Cell 2: Load & xử lý dữ liệu Vietnam ─────────────────────────────────────
path_vn = Path('data/processed/VN_data/full_vietnam_monthly.csv')
df_main = pd.read_csv(path_vn, sep=',')

if 'series' in df_main.columns:
    df_main['series'] = df_main['series'].astype(str).str.strip()
    df_main = df_main[~df_main['series'].str.casefold().eq('total generation')].copy()

df_vn = df_main.rename(columns={
    'generation_twh': 'generation_TWh',
    'precip_mean': 'precipitation',
    'solar_mean': 'solar',
    'humidity_mean': 'humidity',
    'temp_mean': 'temperature'
})

SPARSE_THRESHOLD = 0.60
raw_target = pd.to_numeric(df_vn['generation_TWh'], errors='coerce').fillna(0)
zero_ratio = pd.DataFrame({'series': df_vn['series'], 'raw_target': raw_target}).groupby('series')['raw_target'].apply(lambda s: (s == 0).mean())
sparse_series = zero_ratio[zero_ratio >= SPARSE_THRESHOLD].index.tolist()
if sparse_series:
    df_vn = df_vn[~df_vn['series'].isin(sparse_series)].copy()

df_vn['generation_TWh'] = pd.to_numeric(df_vn['generation_TWh'], errors='coerce').fillna(0).replace(0, 1e-4).clip(lower=1e-4)

if LOG_TRANSFORM:
    df_vn['generation_TWh'] = np.log1p(df_vn['generation_TWh'])

df_vn['date'] = pd.to_datetime(df_vn['date'])
df_vn = df_vn.sort_values(['entity', 'series', 'date']).reset_index(drop=True)

min_date = df_vn['date'].min()
df_vn['time_idx'] = ((df_vn['date'].dt.year - min_date.year) * 12 + (df_vn['date'].dt.month - min_date.month)).astype(int)
df_vn['month'] = df_vn['date'].dt.month.astype(str)
df_vn['month_sin'] = np.sin(2 * np.pi * df_vn['date'].dt.month / 12.0)
df_vn['month_cos'] = np.cos(2 * np.pi * df_vn['date'].dt.month / 12.0)

def create_weather_rolling(group):
    g = group.copy().sort_values('date')
    for w in [6, 12]:
        g[f'precip_roll{w}'] = g['precipitation'].rolling(w, min_periods=1).mean()
        g[f'temp_roll{w}'] = g['temperature'].rolling(w, min_periods=1).mean()
    return g

df_vn = df_vn.groupby(['entity', 'series'], group_keys=False).apply(create_weather_rolling)

sys_hydro = df_vn[df_vn['series'] == 'Hydro'][['date', 'generation_TWh']].rename(columns={'generation_TWh': 'sys_hydro_gen'})
sys_coal = df_vn[df_vn['series'] == 'Coal'][['date', 'generation_TWh']].rename(columns={'generation_TWh': 'sys_coal_gen'})
sys_hydro = sys_hydro.groupby('date')['sys_hydro_gen'].mean().reset_index()
sys_coal = sys_coal.groupby('date')['sys_coal_gen'].mean().reset_index()

df_vn = df_vn.merge(sys_hydro, on='date', how='left')
df_vn = df_vn.merge(sys_coal, on='date', how='left')
df_vn['sys_hydro_gen'] = df_vn['sys_hydro_gen'].fillna(0)
df_vn['sys_coal_gen'] = df_vn['sys_coal_gen'].fillna(0)

def create_rolling_only(group):
    g = group.copy().sort_values('date')
    for w in [3, 6, 12]:
        g[f'target_roll_mean_{w}'] = g['generation_TWh'].shift(1).rolling(w, min_periods=1).mean()
        g[f'target_roll_std_{w}']  = g['generation_TWh'].shift(1).rolling(w, min_periods=1).std().fillna(0)

    if 'sys_hydro_gen' in g.columns:
        g['sys_hydro_shift1'] = g['sys_hydro_gen'].shift(1).fillna(0)
        g['sys_hydro_roll3_mean'] = g['sys_hydro_gen'].shift(1).rolling(3, min_periods=1).mean().fillna(0)
    if 'sys_coal_gen' in g.columns:
        g['sys_coal_shift1'] = g['sys_coal_gen'].shift(1).fillna(0)
        g['sys_coal_roll3_mean'] = g['sys_coal_gen'].shift(1).rolling(3, min_periods=1).mean().fillna(0)
    return g

df_vn = df_vn.groupby(['entity', 'series'], group_keys=False).apply(create_rolling_only)

roll_cols = [c for c in df_vn.columns if 'roll' in c or 'shift' in c]
for c in roll_cols:
    df_vn[c] = df_vn[c].replace(0, 1e-4).clip(lower=1e-4)
    df_vn[c] = df_vn.groupby(['entity', 'series'])[c].transform(lambda x: x.ffill())
    df_vn[c] = df_vn[c].fillna(1e-4)

exo_cols = [
    'precipitation', 'solar', 'humidity', 'temperature',
    'IPI_Value', 'CPI_Value', 'GDP_VND_Trillion', 'Oil_Price',
    'FDI_Disbursed_Monthly(bilionUSD)', 'FDI_Registered_Monthly(bilionUSD)'
]
for c in exo_cols:
    if c in df_vn.columns:
        df_vn[c] = df_vn.groupby(['entity', 'series'])[c].transform(lambda x: x.ffill().bfill())
        df_vn[c] = df_vn[c].fillna(0)
        q01, q99 = df_vn[c].quantile([0.01, 0.99])
        df_vn[c] = df_vn[c].clip(lower=q01, upper=q99)

for c in ['GDP_VND_Trillion', 'Oil_Price', 'FDI_Disbursed_Monthly(bilionUSD)', 'FDI_Registered_Monthly(bilionUSD)']:
    if c in df_vn.columns:
        df_vn[c] = np.log1p(pd.to_numeric(df_vn[c], errors='coerce').fillna(0).clip(lower=0))

print(f'✅ Khởi tạo chuỗi thành công. Kích thước dữ liệu gốc: {len(df_vn)}')


✅ Khởi tạo chuỗi thành công. Kích thước dữ liệu gốc: 360


In [29]:
# ── Cell 3: Tạo TimeSeriesDataSet ─────────────────────────────────────────────
max_encoder_length    = 36
max_prediction_length = 12
batch_size            = 32

min_len = max_encoder_length + max_prediction_length
series_len = df_vn.groupby(['entity', 'series'])['time_idx'].nunique().reset_index(name='n')
valid_groups = series_len.loc[series_len['n'] >= min_len, ['entity', 'series']]
df_vn_valid = df_vn.merge(valid_groups, on=['entity', 'series'], how='inner').copy()
df_vn_valid = df_vn_valid.reset_index(drop=True)

if df_vn_valid.empty:
    raise ValueError('No series long enough!')

training_cutoff = df_vn_valid['time_idx'].max() - max_prediction_length

known_candidates  = [
    'time_idx', 'month_sin', 'month_cos', 'precipitation', 'solar', 'humidity', 'temperature',
    'precip_roll6', 'precip_roll12', 'temp_roll6', 'temp_roll12'
]
unknown_candidates = [
    'generation_TWh',
    'target_roll_mean_3', 'target_roll_mean_6', 'target_roll_mean_12',
    'target_roll_std_3',  'target_roll_std_6',  'target_roll_std_12',
    'sys_hydro_shift1', 'sys_hydro_roll3_mean',
    'sys_coal_shift1', 'sys_coal_roll3_mean',
    'IPI_Value', 'CPI_Value', 'GDP_VND_Trillion',
    'Oil_Price',
    'FDI_Disbursed_Monthly(bilionUSD)', 'FDI_Registered_Monthly(bilionUSD)',
]

known_reals   = [c for c in known_candidates   if c in df_vn_valid.columns]
unknown_reals = [c for c in unknown_candidates if c in df_vn_valid.columns]

# === [FIX] PHỤC HỒI SOFTPLUS ĐỂ KHỚP VỚI PRETRAINED MODEL ===
normalizer = GroupNormalizer(
    groups=['entity', 'series'],
    center=True,
    transformation='softplus'
)

training = TimeSeriesDataSet(
    df_vn_valid[lambda x: x.time_idx <= training_cutoff],
    time_idx='time_idx',
    target='generation_TWh',
    group_ids=['entity', 'series'],
    min_encoder_length=12,         # === [FIX] Hạ Min Encoder xuống 12 để tăng lượng Samples ===
    max_encoder_length=max_encoder_length,
    min_prediction_length=max_prediction_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=['entity', 'series'],
    time_varying_known_categoricals=['month'],
    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=unknown_reals,
    target_normalizer=normalizer,
    lags={'generation_TWh': [1, 2, 3, 6, 12]},
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    df_vn_valid,
    min_prediction_idx=training_cutoff + 1,
    stop_randomization=True,
)

train_loader = training.to_dataloader(train=True,  batch_size=batch_size,     num_workers=0)
val_loader   = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

print(f'✅ Số lượng Mẫu học tập (Windows) – train: {len(training)}, val: {len(validation)}')


✅ Số lượng Mẫu học tập (Windows) – train: 245, val: 125


In [30]:
# ── Cell 4: Khởi tạo model & load pretrained ─────────────────────────────────
os.makedirs('checkpoint', exist_ok=True)
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'

tft_vn = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-4,
    hidden_size=128,
    attention_head_size=4,
    dropout=0.25,                  # === [FIX] Nâng Dropout 0.15 -> 0.25 chống Overfitting ===
    hidden_continuous_size=64,
    loss=QuantileLoss(),
    log_interval=20,
    reduce_on_plateau_patience=5,
    optimizer='adamw',
    weight_decay=1e-2,             # === [FIX] Nâng Weight Decay 1e-3 -> 1e-2 ===
)

n_t = n_m = p_t = p_m = 0
if os.path.exists(CHECKPOINT_PATH):
    pretrained_dict = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)['state_dict']
    model_dict = tft_vn.state_dict()
    matched_dict = {k: v for k, v in pretrained_dict.items()
                    if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(matched_dict)
    tft_vn.load_state_dict(model_dict)
    n_t, n_m = len(model_dict), len(matched_dict)
    p_t = sum(v.numel() for v in model_dict.values())
    p_m = sum(v.numel() for v in matched_dict.values())
    print(f'🔁 Tensors: {n_m}/{n_t} ({n_m/max(n_t,1):.1%}) | Params: {p_m/1e3:.1f}k/{p_t/1e3:.1f}k ({p_m/max(p_t,1):.1%})')
else:
    print(f'⚠️ {CHECKPOINT_PATH} not found – training from scratch')

🔁 Tensors: 566/868 (65.2%) | Params: 1655.9k/2246.6k (73.7%)


In [31]:
# ── Cell 5: Stage 1 – Warm-up (frozen backbone) ───────────────────────────────
print('== STAGE 1: Warm-up (frozen backbone) ==')
FREEZE_LAYERS = ['lstm_encoder', 'lstm_decoder', 'multihead_attn', 'post_attn_norm']
for name, param in tft_vn.named_parameters():
    if any(f in name for f in FREEZE_LAYERS):
        param.requires_grad = False
    else:
        param.requires_grad = True

tft_vn.hparams.learning_rate = 1e-4
cb_es_s1 = EarlyStopping(monitor='val_loss', min_delta=1e-5, patience=8, mode='min')
cb_ckpt_s1 = ModelCheckpoint(monitor='val_loss', mode='min', filename='tft_vn_s1-{epoch:02d}', save_top_k=1)
trainer_s1 = pl.Trainer(max_epochs=15, accelerator=accelerator, devices=1, gradient_clip_val=0.5, callbacks=[cb_es_s1, cb_ckpt_s1], enable_model_summary=False)
trainer_s1.fit(tft_vn, train_dataloaders=train_loader, val_dataloaders=val_loader)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


== STAGE 1: Warm-up (frozen backbone) ==


Output()

`Trainer.fit` stopped: `max_epochs=15` reached.


In [32]:
# ── Cell 6: Stage 2 – Partial unfreeze (LSTM thaw) ───────────────────────────
print('== STAGE 2: Partial unfreeze (LSTM thaw) ==')
if cb_ckpt_s1.best_model_path and os.path.exists(cb_ckpt_s1.best_model_path):
    ckpt = torch.load(cb_ckpt_s1.best_model_path, map_location='cpu', weights_only=False)
    tft_vn.load_state_dict(ckpt['state_dict'])

STILL_FROZEN = ['multihead_attn', 'post_attn_norm']
for name, param in tft_vn.named_parameters():
    param.requires_grad = not any(f in name for f in STILL_FROZEN)

tft_vn.hparams.learning_rate = 4e-5
cb_es_s2 = EarlyStopping(monitor='val_loss', min_delta=1e-5, patience=10, mode='min')
cb_ckpt_s2 = ModelCheckpoint(monitor='val_loss', mode='min', filename='tft_vn_s2-{epoch:02d}', save_top_k=1)
trainer_s2 = pl.Trainer(max_epochs=30, accelerator=accelerator, devices=1, gradient_clip_val=0.5, callbacks=[cb_es_s2, cb_ckpt_s2], enable_model_summary=False)
trainer_s2.fit(tft_vn, train_dataloaders=train_loader, val_dataloaders=val_loader)


== STAGE 2: Partial unfreeze (LSTM thaw) ==


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

In [33]:
# ── Cell 7: Stage 3 – Full finetune ──────────────────────────────────────────
print('== STAGE 3: Full finetune (all layers) ==')
if cb_ckpt_s2.best_model_path and os.path.exists(cb_ckpt_s2.best_model_path):
    ckpt = torch.load(cb_ckpt_s2.best_model_path, map_location='cpu', weights_only=False)
    tft_vn.load_state_dict(ckpt['state_dict'])

for param in tft_vn.parameters():
    param.requires_grad = True

tft_vn.hparams.learning_rate = 1e-5
cb_es_s3 = EarlyStopping(monitor='val_loss', min_delta=5e-6, patience=12, mode='min')
cb_ckpt_s3 = ModelCheckpoint(dirpath='checkpoint', monitor='val_loss', mode='min', filename='tft_vn_best_final-{epoch:02d}', save_top_k=1)
trainer_s3 = pl.Trainer(max_epochs=40, accelerator=accelerator, devices=1, gradient_clip_val=0.5, callbacks=[cb_es_s3, cb_ckpt_s3], enable_model_summary=False)
trainer_s3.fit(tft_vn, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer_vn = trainer_s3


== STAGE 3: Full finetune (all layers) ==


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

In [34]:
# ── Cell 8: Đánh giá (Fixed evaluation) ──────────────────────────────────────
if cb_ckpt_s3.best_model_path and os.path.exists(cb_ckpt_s3.best_model_path):
    print(f'Load Final: {cb_ckpt_s3.best_model_path}')
    try: tft_vn = TemporalFusionTransformer.load_from_checkpoint(cb_ckpt_s3.best_model_path, weights_only=False)
    except: tft_vn = TemporalFusionTransformer.load_from_checkpoint(cb_ckpt_s3.best_model_path)

torch.set_grad_enabled(False)
tft_vn.eval()

raw_pred_obj = tft_vn.predict(val_loader, mode='raw', return_x=True)
raw_predictions = raw_pred_obj.output if hasattr(raw_pred_obj, 'output') else raw_pred_obj[0] if isinstance(raw_pred_obj, tuple) else raw_pred_obj
x_plot = getattr(raw_pred_obj, 'x', None) if hasattr(raw_pred_obj, 'x') else raw_pred_obj[1] if isinstance(raw_pred_obj, tuple) else None

point_pred_obj = tft_vn.predict(val_loader, mode='prediction')
point_predictions = point_pred_obj.output if hasattr(point_pred_obj, 'output') else point_pred_obj

actuals_list = []
for _, y in iter(val_loader):
    target = y[0] if isinstance(y, (list, tuple)) else y
    actuals_list.append(target.reshape(-1))
act_log = torch.cat(actuals_list, dim=0)
prd_log = point_predictions.detach().cpu().reshape(-1)
n = min(act_log.shape[0], prd_log.shape[0])
act_log, prd_log = act_log[:n], prd_log[:n]

ok = torch.isfinite(act_log) & torch.isfinite(prd_log)
act_log, prd_log = act_log[ok], prd_log[ok]

if LOG_TRANSFORM:
    act_twh = torch.expm1(act_log).clamp(min=0)
    prd_twh = torch.expm1(prd_log).clamp(min=0)
else:
    act_twh, prd_twh = act_log, prd_log

mae  = (act_twh - prd_twh).abs().mean().item()
rmse = torch.sqrt(((act_twh - prd_twh)**2).mean()).item()
wape = ((act_twh - prd_twh).abs().sum() / (act_twh.abs().sum() + 1e-8)).item() * 100
r2   = r2_score(act_twh.numpy(), prd_twh.numpy())

print(f'\n--- FINAL OVERALL METRICS (TWh scale) ---')
print(f'MAE : {mae:.4f}  (Mục tiêu: < 0.8)')
print(f'RMSE: {rmse:.4f}  (Mục tiêu: ~ 1.1)')
print(f'WAPE: {wape:.4f}%')
print(f'R²  : {r2:.4f}')

# Lưu checkpoints
os.makedirs('checkpoint', exist_ok=True)
trainer_vn.save_checkpoint('checkpoint/tft_vn_model_latest.ckpt')
shutil.copy(cb_ckpt_s3.best_model_path, 'checkpoint/tft_vn_model_best.ckpt')
print('\n✅ Mô hình đã được cập nhật thành công!')


== STAGE 2: Partial unfreeze (LSTM thaw) ==


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

In [35]:
# ── Cell 9: Vẽ biểu đồ ─────────────────────────────────────────────────────
try:
    fig, ax = plt.subplots(figsize=(12, 5))
    if x_plot is not None and raw_predictions is not None:
        _ = tft_vn.plot_prediction(x_plot, raw_predictions, idx=0, add_loss_to_title=True, ax=ax)
        plt.title('VN Transfer Learning – Prediction')
        plt.tight_layout()
        plt.show()
    else:
        print('⚠️ Không thể vẽ biểu đồ vì dữ liệu plot trả về lỗi/trống')
except Exception as e:
    print(f'Plot error: {e}')

Load Final: C:\Users\ADMIN\Downloads\MODEL_TFT\checkpoint\tft_vn_best_final-epoch=00-v2.ckpt


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightn


--- FINAL OVERALL METRICS (TWh scale) ---
MAE : 1.0284  (Mục tiêu: < 0.8)
RMSE: 1.6806  (Mục tiêu: ~ 1.1)
WAPE: 20.0171%
R²  : 0.8802

✅ Mô hình đã được cập nhật thành công!
